# EDEN-04 — Climate forecasting baseline | v1.0
**Working Phase 2 · Notebook 04 of 15 · CPU only**

This notebook uses the verified EDEN-01 climate table to build two simple air-temperature references: persistence (use the latest valid reading) and same-time-yesterday. It evaluates a provisional **one-hour forecast** on the frozen primary and chronological splits. It does not train a machine-learning model or use future measurements.

**Scope note:** the full EDEN SRS is not in the supplied project files. Climate forecasting is the project’s stated direction for the next work area, but the exact phase/notebook assignment and horizon need confirmation against that SRS. The 1-hour horizon is an explicit starter choice, within EDEN-01’s frozen 72-step maximum.

**Input:** attach the EDEN-01 v1.1 `PASS_ETL` result ZIP or its extracted `eden/` folder. **Output:** `EDEN-04-Climate-Forecasting-Baseline-Results-v1.0.zip` with eligible windows, baseline predictions, error summaries, plot, and manifest.

Run cells in order in Kaggle with **CPU, Internet Off**. This notebook does not depend on EDEN-03 masks. Review the output before selecting targets, horizons, or model designs for later notebooks.

## Cell 1 — Set the forecast and resource settings

This cell imports table/file tools and sets one target and forecast horizon. Each row is five minutes apart, so 12 steps means one hour. The horizon is easy to change, but keep it within EDEN-01’s frozen limit and document any change.

In [ ]:
# Import tools for paths, tables, hashes, ZIP files, and a small plot.
from pathlib import Path
import hashlib, json, zipfile, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Kaggle input is read-only; working is where this notebook writes results.
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
OUT_ROOT = WORK_ROOT / 'eden04'
RESULT_NAME = 'EDEN-04-Climate-Forecasting-Baseline-Results-v1.0.zip'

# Use one metadata-backed target and a starter forecast horizon.
TARGET = 'air_temp_c'       # Measured air temperature, degrees Celsius.
HORIZON_STEPS = 12          # 12 five-minute intervals = one hour ahead.
STEPS_PER_DAY = 288         # 24 hours * 60 minutes / 5 minutes per row.

# Recreate only this notebook's own output folder.
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)
for folder in ['data', 'reports']:
    (OUT_ROOT / folder).mkdir(parents=True, exist_ok=True)
print('Target:', TARGET, '| horizon steps:', HORIZON_STEPS, '| minutes:', HORIZON_STEPS * 5)

## Cell 2 — Find and verify the EDEN-01 result

This cell searches the attached Kaggle datasets for the EDEN-01 v1.1 result. It accepts the named ZIP or an extracted folder. It checks the producer, version, successful ETL status, file sizes, and SHA-256 fingerprints before using the data. If more than one matching result is attached, it stops instead of picking one silently.

In [ ]:
# Look for likely named EDEN-01 ZIP files first; otherwise inspect attached ZIP manifests.
zip_candidates = []
for path in INPUT_ROOT.rglob('*.zip'):
    try:
        with zipfile.ZipFile(path) as archive:
            manifest_name = next((n for n in archive.namelist() if n.endswith('/manifest.json')), None)
            if manifest_name:
                candidate = json.loads(archive.read(manifest_name))
                if candidate.get('producer') == 'EDEN-01' and candidate.get('code_version') == '1.1':
                    zip_candidates.append((path, manifest_name))
    except (OSError, zipfile.BadZipFile, json.JSONDecodeError):
        continue

# Search extracted inputs only when no matching EDEN-01 ZIP was found.
folder_candidates = []
if not zip_candidates:
    for path in INPUT_ROOT.rglob('manifest.json'):
        try:
            candidate = json.loads(path.read_text(encoding='utf-8'))
            if candidate.get('producer') == 'EDEN-01' and candidate.get('code_version') == '1.1':
                folder_candidates.append((path, candidate))
        except (OSError, json.JSONDecodeError):
            continue

# Require one unambiguous source, so outputs cannot be mixed across runs.
if len(zip_candidates) + len(folder_candidates) != 1:
    raise RuntimeError(f'Expected one EDEN-01 v1.1 result; found {len(zip_candidates)} ZIPs and {len(folder_candidates)} folders. Remove extra versions or set an explicit input path.')

# Check every file against the upstream receipt and unpack only after verification.
if zip_candidates:
    source_zip, manifest_name = zip_candidates[0]
    with zipfile.ZipFile(source_zip) as archive:
        source_manifest = json.loads(archive.read(manifest_name))
        prefix = manifest_name.rsplit('/', 1)[0]
        assert source_manifest.get('status') == 'PASS_ETL'
        for item in source_manifest['files']:
            member = f"{prefix}/{item['path']}"
            payload = archive.read(member)
            assert len(payload) == item['size'], f'Size mismatch: {member}'
            assert hashlib.sha256(payload).hexdigest() == item['sha256'], f'Hash mismatch: {member}'
        archive.extractall(WORK_ROOT / 'eden01_input')
        SOURCE_ROOT = WORK_ROOT / 'eden01_input' / prefix
else:
    manifest_path, source_manifest = folder_candidates[0]
    SOURCE_ROOT = manifest_path.parent
    assert source_manifest.get('status') == 'PASS_ETL'
    for item in source_manifest['files']:
        file_path = SOURCE_ROOT / item['path']
        assert file_path.stat().st_size == item['size'], f'Size mismatch: {file_path}'
        assert hashlib.sha256(file_path.read_bytes()).hexdigest() == item['sha256'], f'Hash mismatch: {file_path}'

print('Verified EDEN-01:', source_manifest['status'], '| files:', len(source_manifest['files']))

## Cell 3 — Load the climate table and frozen split rules

The climate table is the cleaned five-minute dataset. A **split** is a fixed group reserved for training, validation, or testing. This cell checks unique compartment/time rows, UTC timestamps, the five-minute grid, the target’s measurement unit, and the frozen limit on forecast horizon.

In [ ]:
# Load the exact saved climate table and project split definitions.
climate = pd.read_parquet(SOURCE_ROOT / 'data/climate.parquet')
splits = json.loads((SOURCE_ROOT / 'data/splits.json').read_text(encoding='utf-8'))
schema = json.loads((SOURCE_ROOT / 'data/schema_map.json').read_text(encoding='utf-8'))

# Confirm target metadata, key uniqueness, UTC time, and the agreed sampling interval.
target_info = next(x for x in schema['fields'] if x['canonical'] == TARGET)
assert target_info['unit'] == 'degree_Celsius' and target_info['kind'] == 'measurement'
assert {TARGET, TARGET + '_observed', TARGET + '_qc_ok', 'experiment_id', 'compartment_id', 'ts_utc', 'split', 'chronological_split'} <= set(climate.columns)
assert climate['compartment_id'].astype(str).isin(['3.01','3.02','3.03','3.06','3.07','3.08']).all()
assert not climate.duplicated(['experiment_id','compartment_id','ts_utc']).any()
assert str(climate['ts_utc'].dt.tz) == 'UTC'
assert HORIZON_STEPS <= int(splits['max_horizon_steps'])
assert int(splits['timestep_minutes']) == 5
assert splits['primary']['compartments']['train'] == ['3.01','3.02','3.03','3.06']
assert splits['primary']['compartments']['validation'] == ['3.07']
assert splits['primary']['compartments']['test'] == ['3.08']

# Check each compartment follows the expected regular five-minute grid.
for compartment_id, group in climate.groupby('compartment_id'):
    times = group.sort_values('ts_utc')['ts_utc']
    assert times.diff().dropna().eq(pd.Timedelta(minutes=5)).all(), f'Unexpected time gap: {compartment_id}'

print('Rows:', len(climate), '| compartments:', climate.compartment_id.nunique(), '| target:', TARGET, target_info['unit'])

## Cell 4 — Build leakage-safe one-hour forecast examples

For each forecast start time, the **persistence baseline** repeats the current valid temperature. The **same-time-yesterday baseline** uses the reading from 24 hours before the forecast target. Both target and predictor must pass EDEN-01’s observation and QC flags. Input and target must remain in the same frozen split; embargo rows are excluded. The primary and chronological protocols are calculated separately.

In [ ]:
# Build forecast rows for one split protocol without crossing compartments or split boundaries.
def build_protocol(frame, split_column, protocol_name):
    # Collect eligible windows and coverage counts for this one protocol.
    windows = []
    counts = []
    for compartment_id, original in frame.groupby('compartment_id', sort=True):
        # Sort by UTC and reset row numbers so fixed five-minute shifts are clear.
        group = original.sort_values('ts_utc').reset_index(drop=True).copy()
        # Align the future target exactly HORIZON_STEPS after each forecast origin.
        group['target_time'] = group['ts_utc'].shift(-HORIZON_STEPS)
        group['actual'] = group[TARGET].shift(-HORIZON_STEPS)
        group['future_qc'] = group[TARGET + '_qc_ok'].shift(-HORIZON_STEPS).fillna(False).astype(bool)
        group['future_observed'] = group[TARGET + '_observed'].shift(-HORIZON_STEPS).fillna(False).astype(bool)
        group['target_split'] = group[split_column].shift(-HORIZON_STEPS)
        # The seasonal value is the same five-minute time on the previous day.
        seasonal_lag = STEPS_PER_DAY - HORIZON_STEPS
        group['same_time_yesterday'] = group[TARGET].shift(seasonal_lag)
        group['yesterday_qc'] = group[TARGET + '_qc_ok'].shift(seasonal_lag).fillna(False).astype(bool)
        group['yesterday_observed'] = group[TARGET + '_observed'].shift(seasonal_lag).fillna(False).astype(bool)
        # Record all potential and eligible rows by their frozen origin split.
        for split_name, split_group in group.groupby(split_column, dropna=False):
            if split_name not in ('train', 'validation', 'test'):
                continue
            split_group = split_group.copy()
            split_group['protocol'] = protocol_name
            split_group['compartment_id'] = str(compartment_id)
            counts.append({'protocol': protocol_name, 'split': split_name, 'compartment_id': str(compartment_id), 'candidate_origins': len(split_group)})
        # Keep only origins and future targets in the same train/validation/test group.
        valid = group[split_column].isin(['train', 'validation', 'test'])
        valid &= group[split_column].eq(group['target_split'])
        valid &= group[TARGET + '_qc_ok'].fillna(False).astype(bool)
        valid &= group[TARGET + '_observed'].fillna(False).astype(bool)
        valid &= group['future_qc'] & group['future_observed'] & group['actual'].notna()
        # Both baseline predictions must be available from valid past measurements.
        valid &= group['yesterday_qc'] & group['yesterday_observed'] & group['same_time_yesterday'].notna()
        keep = group.loc[valid, ['experiment_id','compartment_id','ts_utc','target_time',split_column,TARGET,'actual','same_time_yesterday']].copy()
        keep = keep.rename(columns={split_column: 'split', TARGET: 'persistence'})
        keep['protocol'] = protocol_name
        keep['persistence_prediction'] = keep['persistence']
        keep['seasonal_prediction'] = keep['same_time_yesterday']
        windows.append(keep)
    # Join compartment windows into one table with explicit protocol labels.
    result = pd.concat(windows, ignore_index=True) if windows else pd.DataFrame()
    return result, pd.DataFrame(counts)

# Score these as two independent evaluation protocols; never combine their split labels.
primary_windows, primary_counts = build_protocol(climate, 'split', 'primary_compartment')
chron_windows, chron_counts = build_protocol(climate, 'chronological_split', 'chronological_time')
predictions = pd.concat([primary_windows, chron_windows], ignore_index=True)
counts = pd.concat([primary_counts, chron_counts], ignore_index=True)
assert not predictions.empty, 'No eligible forecast examples; inspect target QC and split files.'
assert predictions.groupby('protocol').split.nunique().ge(2).all(), 'A protocol is missing validation or test windows.'
predictions.to_csv(OUT_ROOT / 'data/forecast_baseline_predictions.csv', index=False)
counts.to_csv(OUT_ROOT / 'reports/candidate_origin_counts.csv', index=False)
print(predictions.groupby(['protocol','split']).size().rename('eligible_windows'))

## Cell 5 — Compare the two fixed baseline forecasts

This cell calculates **MAE** (average absolute error), **RMSE** (larger errors count more), and **bias** (average prediction minus actual). It reports validation and test separately for each protocol and also keeps per-compartment scores. These baselines have no fitted parameters, so the rules are fixed before scoring; test scores are still descriptive results for this dataset, not proof of future performance.

In [ ]:
# Calculate error measures for one set of forecast rows.
def score(group, model_name, prediction_column):
    # Keep only rows with both an actual value and a prediction.
    valid_rows = group[['actual', prediction_column]].dropna()
    error = valid_rows[prediction_column] - valid_rows['actual']
    # Return sample count and three familiar error summaries in degrees Celsius.
    return {'model': model_name, 'n': len(error),
            'mae_c': float(error.abs().mean()) if len(error) else np.nan,
            'rmse_c': float(np.sqrt(np.mean(error ** 2))) if len(error) else np.nan,
            'bias_c': float(error.mean()) if len(error) else np.nan}

# Summarize each fixed rule by protocol and split.
metric_rows = []
for (protocol, split), group in predictions.groupby(['protocol','split']):
    for model, column in [('persistence','persistence_prediction'), ('same_time_yesterday','seasonal_prediction')]:
        metric_rows.append({'protocol': protocol, 'split': split, **score(group, model, column)})
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(OUT_ROOT / 'reports/baseline_metrics.csv', index=False)

# Keep per-compartment numbers visible so a single compartment cannot hide variation.
compartment_rows = []
for (protocol, split, compartment_id), group in predictions.groupby(['protocol','split','compartment_id']):
    for model, column in [('persistence','persistence_prediction'), ('same_time_yesterday','seasonal_prediction')]:
        compartment_rows.append({'protocol': protocol, 'split': split, 'compartment_id': str(compartment_id), **score(group, model, column)})
compartment_metrics = pd.DataFrame(compartment_rows)
compartment_metrics.to_csv(OUT_ROOT / 'reports/compartment_metrics.csv', index=False)
display(metrics.sort_values(['protocol','split','model']))

## Cell 6 — Plot held-out examples and save coverage notes

The plot shows a short validation segment for one compartment: observed future temperature and both baseline predictions. It is an example, not a selection tool. The coverage report records the number of valid forecast windows in each frozen split and the number of rows excluded by missing or failed QC checks.

In [ ]:
# Choose the first chronological validation compartment with eligible predictions.
example_group = predictions[(predictions.protocol == 'chronological_time') & (predictions.split == 'validation')]
assert not example_group.empty, 'No chronological validation examples are available.'
example_compartment = sorted(example_group.compartment_id.unique())[0]
example = example_group[example_group.compartment_id == example_compartment].sort_values('target_time').head(288)

# Plot one day at most, keeping time in UTC and units in Celsius.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(example.target_time, example.actual, label='Observed target', linewidth=1.4)
ax.plot(example.target_time, example.persistence_prediction, label='Persistence', alpha=0.8)
ax.plot(example.target_time, example.seasonal_prediction, label='Same time yesterday', alpha=0.8)
ax.set_title(f'EDEN-04 one-hour air-temperature baseline | compartment {example_compartment}')
ax.set_ylabel('Air temperature (°C)')
ax.set_xlabel('Target time (UTC)')
ax.legend()
fig.tight_layout()
fig.savefig(OUT_ROOT / 'reports/validation_example.png', dpi=130)
plt.close(fig)

# Record what this starter notebook does and does not establish.
summary = {
    'producer': 'EDEN-04', 'code_version': '1.0', 'scope_status': 'PROVISIONAL_PENDING_FULL_SRS',
    'input_status': source_manifest['status'], 'target': TARGET, 'target_unit': 'degree_Celsius',
    'horizon_steps': HORIZON_STEPS, 'horizon_minutes': HORIZON_STEPS * 5,
    'models': ['persistence','same_time_yesterday'], 'model_fitting': False,
    'future_weather_used': False, 'protocols_reported_separately': True,
    'primary_validation_rows': int(len(primary_windows[primary_windows.split == 'validation'])),
    'primary_test_rows': int(len(primary_windows[primary_windows.split == 'test'])),
    'chronological_validation_rows': int(len(chron_windows[chron_windows.split == 'validation'])),
    'chronological_test_rows': int(len(chron_windows[chron_windows.split == 'test'])),
    'pending': ['Confirm notebook/phase allocation and forecast horizon against full SRS',
                'Review results before selecting further targets or fitting a model',
                'No uncertainty interval or model-training result is produced in this baseline notebook']
}
(OUT_ROOT / 'reports/forecast_baseline_report.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

## Cell 7 — Acceptance checks and named output ZIP

The acceptance gate checks the named input status, exact forecast horizon, available validation/test windows, fixed baseline columns, and output files. It records a provisional status: successful baseline generation does not mean the SRS is complete or that a forecasting model is accurate. The output ZIP is saved outside its source folder.

In [ ]:
# Require both baseline rules and separate split protocols in the prediction file.
assert {'persistence_prediction','seasonal_prediction','protocol','split','actual'} <= set(predictions.columns)
assert set(predictions.protocol) == {'primary_compartment','chronological_time'}
assert predictions.groupby('protocol').split.apply(lambda x: {'validation','test'} <= set(x)).all()
assert metrics[['mae_c','rmse_c','bias_c']].notna().all().all()

# Inventory every result file and calculate a SHA-256 fingerprint.
result_files = sorted(path for path in OUT_ROOT.rglob('*') if path.is_file() and path.name != 'manifest.json')
manifest = {
    'schema_version': '1.0', 'producer': 'EDEN-04', 'code_version': '1.0',
    'status': 'PASS_BASELINE_EVAL_PROVISIONAL',
    'scope_status': 'PROVISIONAL_PENDING_FULL_SRS',
    'generated_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'upstream_manifest_sha256': hashlib.sha256((SOURCE_ROOT / 'manifest.json').read_bytes()).hexdigest(),
    'target': TARGET, 'horizon_steps': HORIZON_STEPS,
    'protocols_reported_separately': True, 'model_fitting': False,
    'files': []
}
for path in result_files:
    manifest['files'].append({'path': path.relative_to(OUT_ROOT).as_posix(), 'size': path.stat().st_size,
                              'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
(OUT_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

# Save one portable, versioned ZIP for the next handoff.
result_zip = WORK_ROOT / RESULT_NAME
if result_zip.exists():
    result_zip.unlink()
with zipfile.ZipFile(result_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
    for path in sorted(p for p in OUT_ROOT.rglob('*') if p.is_file()):
        archive.write(path, arcname=f'eden/{path.relative_to(OUT_ROOT).as_posix()}')
print('Status:', manifest['status'])
print('Download:', result_zip)

## Run, review, and hand off

1. In Kaggle, create a notebook with **CPU**, **Internet Off**, and attach the EDEN-01 v1.1 `PASS_ETL` result ZIP (or extracted output folder).
2. Run all cells in order. If more than one EDEN-01 v1.1 output is attached, remove extras and rerun from a clean session.
3. Download `EDEN-04-Climate-Forecasting-Baseline-Results-v1.0.zip` from `/kaggle/working/` and upload it here.
4. Review `reports/baseline_metrics.csv`, `reports/compartment_metrics.csv`, and `reports/validation_example.png` before planning a learned model.

**Troubleshooting:** A missing PyArrow import means Pandas cannot read Parquet; use Kaggle's standard Python image or install PyArrow in a separate Internet-enabled session, then restart and rerun. A hash failure means reattach the original EDEN-01 output. An empty validation/test group means inspect the split definitions and QC coverage; do not loosen the frozen split or QC masks just to make the gate pass.

**Limits:** this is a provisional baseline notebook because the full SRS is missing. It forecasts only `air_temp_c` at one hour ahead and uses no controls, weather forecasts, imagery, fitted model, or uncertainty interval. Source split protocols are kept separate; a later model must compare fairly against these baselines and define any extra covariates before using them.

Source: Maree et al. (2025), [4th Autonomous Greenhouse Challenge: Dwarf Tomato Timeseries and Images, v1](https://doi.org/10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1), CC BY 4.0.